# 1. Getting Started

This notebook gives you the shortest complete introduction to `KnottedGraph`.

By the end, you will have:

- constructed a simple three-dimensional surface neighborhood;
- inspected the embedded spatial graph carried by that surface;
- chosen a planar projection;
- inspected the corresponding PD code; and
- computed the Yamada polynomial.

The basic workflow is

$$
\text{geometric object}
\longrightarrow
G\subset\mathbb{R}^3
\longrightarrow
D(G)
\longrightarrow
\operatorname{PD}(G)
\longrightarrow
\Upsilon(G;Y).
$$

A repulsive-curve layout can be inserted between the spatial graph and the
projection when you want a cleaner geometric representative.  The example is synthetic so you can see the topology workflow without first
preparing a domain-specific dataset; the application notebooks explain those routes in more detail.

## 1.1 Install KnottedGraph

This notebook documents the **0.2 development API**. Until that release is
published, install it from the `Latest_Workplace` branch rather than from the
older package currently available on PyPI:

```bash
git clone --branch Latest_Workplace --single-branch https://github.com/sarinstein-yan/KnottedGraph.git
cd KnottedGraph
uv sync --extra viz --extra notebook
uv run jupyter lab
```

The `viz` extra supplies Plotly for the interactive figures in this notebook;
`notebook` supplies JupyterLab. After 0.2 is released, the equivalent wheel
installation will be `pip install "knotted_graph[viz,notebook]"`. See the
website installation and troubleshooting pages for the complete extras matrix.

The native C++ Yamada backend is optional for correctness but strongly recommended
for performance. The verification cell below reports whether it is active.

## 1.2 Verify the installation

Run this cell **before the tutorial**. It checks the active Python interpreter,
where `knotted_graph` is imported from, core dependencies, and the compiled Yamada
backend. An editable install may correctly report a Python file under `src/`; what
matters is that the environment was installed with `pip -e` and the native extension
is discoverable through that installation.

For a performance-ready source installation, the desired backend result is:

```text
Native Yamada backend: True
Native import error: None
```

If the backend is `False`, Yamada results remain exact through the Python fallback,
but larger calculations can be substantially slower.

The setup accepts either a source checkout or an installed 0.2 package. It
never inserts a guessed `/src` directory when neither one is available.

In [ ]:
from pathlib import Path
import importlib.util
import os
import sys
import tempfile

candidate = Path.cwd().resolve()
while not (candidate / "src" / "knotted_graph").exists() and candidate != candidate.parent:
    candidate = candidate.parent

if (candidate / "src" / "knotted_graph").exists():
    PROJECT_ROOT = candidate
    SRC_ROOT = PROJECT_ROOT / "src"
    if str(SRC_ROOT) not in sys.path:
        sys.path.insert(0, str(SRC_ROOT))
    installation_mode = f"source checkout: {PROJECT_ROOT}"
elif importlib.util.find_spec("knotted_graph") is not None:
    PROJECT_ROOT = None
    installation_mode = "installed package"
else:
    raise RuntimeError(
        "KnottedGraph was not found. Run this notebook inside a source checkout "
        "or install the current 0.2 development package first."
    )

missing_api = [
    name
    for name in ("knotted_graph.core", "knotted_graph.projection")
    if importlib.util.find_spec(name) is None
]
if missing_api:
    raise RuntimeError(
        "The installed knotted_graph package is missing the current 0.2 API "
        f"modules: {', '.join(missing_api)}. Install the Latest_Workplace source branch."
    )
import knotted_graph
installation_mode += f" (version {knotted_graph.__version__})"

os.environ.setdefault(
    "MPLCONFIGDIR",
    str(Path(tempfile.gettempdir()) / "knottedgraph-mpl"),
)

from knotted_graph.invariants.yamada.native import (
    native_available,
    native_import_error,
)

print(f"KnottedGraph mode = {installation_mode}")
print("Python executable:", sys.executable)
print("KnottedGraph:", Path(knotted_graph.__file__).resolve())
print("Native Yamada backend:", native_available())
print("Native import error:", native_import_error())

print("\nDependencies:")
for package in ["numpy", "networkx", "sympy", "shapely", "matplotlib", "plotly"]:
    print(f"{package:12s}: {importlib.util.find_spec(package) is not None}")

if native_available():
    import knotted_graph.invariants.yamada._yamada_native as _yamada_native
    print("Native extension:", Path(_yamada_native.__file__).resolve())
else:
    print("WARNING: exact Python Yamada fallback is active; high-crossing calculations may be slow.")


In [ ]:
import math
import time
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
import sympy as sp
from IPython.display import Math, display

from knotted_graph.projection import (
    compute_yamada_polynomial,
    sample_projections,
    select_projection,
)
from knotted_graph.visualization import plot_3D_graph_plotly

BLUE = "#1f77b4"
RED = "#d62728"
CAMERA = dict(eye=dict(x=4.0, y=4.0, z=3.0))
pio.renderers.default = "notebook_connected"
Y = sp.Symbol("Y")
kx, ky, kz = sp.symbols("k_x k_y k_z", real=True)


def axis_style():
    return dict(
        visible=True,
        title="",
        showticklabels=False,
        showbackground=False,
        showgrid=False,
        zeroline=False,
        showline=True,
        linecolor="black",
        linewidth=2,
    )


def apply_kg_layout(fig, *, width=760, height=620):
    fig.update_layout(
        title=None,
        width=width,
        height=height,
        margin=dict(l=0, r=0, t=0, b=0),
        scene=dict(
            xaxis=axis_style(),
            yaxis=axis_style(),
            zaxis=axis_style(),
            aspectmode="data",
            camera=CAMERA,
        ),
    )
    return fig


def plot_surface_polydata(surface, *, opacity=0.58):
    mesh = surface.triangulate()
    faces = mesh.faces.reshape(-1, 4)[:, 1:]
    pts = mesh.points
    fig = go.Figure(
        go.Mesh3d(
            x=pts[:, 0],
            y=pts[:, 1],
            z=pts[:, 2],
            i=faces[:, 0],
            j=faces[:, 1],
            k=faces[:, 2],
            color=BLUE,
            opacity=opacity,
        )
    )
    return apply_kg_layout(fig)


def plot_points_3d(points, *, size=3):
    points = np.asarray(points)
    fig = go.Figure(
        go.Scatter3d(
            x=points[:, 0],
            y=points[:, 1],
            z=points[:, 2],
            mode="markers",
            marker=dict(size=size, color=BLUE),
        )
    )
    return apply_kg_layout(fig)


def plot_graph_kg(graph):
    return apply_kg_layout(plot_3D_graph_plotly(graph))


def print_upsilon(label, expr):
    print(f"Upsilon({label}; Y) = {sp.expand(expr)}")


def display_bloch_vector(label, components):
    display(Math(label + r"=" + sp.latex(sp.Matrix(components))))


print("shared plotting and notation helpers ready")

def plot_projection_diagram(
    projection,
    *,
    title=None,
    annotate=False,
    figsize=(5.4, 4.6),
    edge_color=BLUE,
    vertex_color=RED,
    line_width=2.4,
    vertex_size=64,
    gap_fraction=0.015,
):
    """Draw blue edges, red vertices, and explicit over/under crossing gaps."""
    fig, ax = plt.subplots(figsize=figsize)

    arcs_by_id = {
        arc.id: arc
        for arc in projection.arcs
    }

    all_xy = []

    # Draw the complete projected diagram first.
    for arc in projection.arcs:
        x, y = arc.line.xy
        ax.plot(
            x,
            y,
            color=edge_color,
            linewidth=line_width,
            solid_capstyle="round",
            zorder=1,
        )
        all_xy.extend(zip(x, y))

        if annotate:
            midpoint = arc.line.interpolate(
                0.5,
                normalized=True,
            )
            ax.text(
                midpoint.x,
                midpoint.y,
                f"a{arc.id}",
                fontsize=8,
                color=edge_color,
                zorder=7,
            )

    if all_xy:
        xy = np.asarray(
            all_xy,
            dtype=float,
        )
        diagram_span = max(
            float(np.ptp(xy[:, 0])),
            float(np.ptp(xy[:, 1])),
            1.0,
        )
    else:
        diagram_span = 1.0

    gap_radius = (
        gap_fraction
        * diagram_span
    )

    for crossing in projection.crossings:
        try:
            ordered_arcs = list(
                crossing.ccw_ordered_arcs
            )
        except Exception:
            ordered_arcs = []

        if len(ordered_arcs) != 4:
            continue

        cx = crossing.point.x
        cy = crossing.point.y

        # Temporarily erase both strands around the crossing.
        ax.add_patch(
            plt.Circle(
                (cx, cy),
                gap_radius,
                facecolor="white",
                edgecolor="none",
                zorder=4,
            )
        )

        # In KnottedGraph's crossing order, entries 0 and 2 form
        # the over-strand. Redraw those two half-arcs continuously.
        for arc_id in (
            ordered_arcs[0],
            ordered_arcs[2],
        ):
            arc = arcs_by_id.get(
                arc_id
            )

            if (
                arc is None
                or arc.line.length <= 0
            ):
                continue

            probe_distance = min(
                2.4 * gap_radius,
                0.45 * arc.line.length,
            )

            if (
                arc.start_type == "x"
                and arc.start_id
                == crossing.id
            ):
                outside_point = (
                    arc.line.interpolate(
                        probe_distance
                    )
                )

            elif (
                arc.end_type == "x"
                and arc.end_id
                == crossing.id
            ):
                outside_point = (
                    arc.line.interpolate(
                        max(
                            arc.line.length
                            - probe_distance,
                            0.0,
                        )
                    )
                )

            else:
                continue

            ax.plot(
                [cx, outside_point.x],
                [cy, outside_point.y],
                color=edge_color,
                linewidth=line_width,
                solid_capstyle="round",
                zorder=5,
            )

        if annotate:
            ax.annotate(
                f"x{crossing.id}",
                (cx, cy),
                xytext=(5, 5),
                textcoords="offset points",
                fontsize=8,
                color="black",
                zorder=8,
            )

    # Rigid graph vertices are always red.
    for vertex in projection.vertices:
        ax.scatter(
            *vertex.point.xy,
            color=vertex_color,
            s=vertex_size,
            zorder=6,
        )

        if annotate:
            ax.annotate(
                f"v{vertex.id}",
                vertex.point.xy,
                xytext=(5, -10),
                textcoords="offset points",
                fontsize=8,
                color=vertex_color,
                zorder=8,
            )

    ax.set_aspect("equal")
    ax.axis("off")

    if title:
        ax.set_title(title)

    plt.show()
    return fig, ax

## 1.3 Build a simple three-dimensional example

Start with a finite-thickness neighborhood of a trivalent $K_4$ spatial graph.
The tube pieces represent edge neighborhoods and the four branch regions
represent degree-three vertices.

This is a useful first example because you can see both objects that matter:

- the **finite-thickness geometry** you might obtain from a physical or geometric
  problem; and
- the **embedded graph spine** that is passed to the topology tools.

Run the next cell to construct both representations.


In [ ]:
def trivalent_k4_spine(samples=90, amplitude=0.75):
    vertices = {
        "a": np.array([-1.15, -0.78, -0.38]),
        "b": np.array([1.18, -0.64, 0.30]),
        "c": np.array([0.86, 0.95, -0.26]),
        "d": np.array([-0.88, 0.84, 0.52]),
    }
    edge_specs = [
        ("a", "b", "ab", np.array([0.00, 0.90, 0.70]), 0.0),
        ("a", "c", "ac", np.array([0.35, -0.15, 1.00]), 1.1),
        ("a", "d", "ad", np.array([0.95, 0.15, -0.25]), 2.2),
        ("b", "c", "bc", np.array([-0.90, 0.25, 0.35]), 0.7),
        ("b", "d", "bd", np.array([-0.20, 1.00, -0.60]), 1.7),
        ("c", "d", "cd", np.array([0.10, -0.90, -0.85]), 2.8),
    ]

    s = np.linspace(0.0, 1.0, samples)
    graph = nx.MultiGraph()
    for vertex_id, pos in vertices.items():
        graph.add_node(vertex_id, pos=pos)

    for u, v, key, bend, phase in edge_specs:
        start = vertices[u]
        end = vertices[v]
        chord = end - start
        bend = bend / np.linalg.norm(bend)
        side = np.cross(chord, bend)
        side = side / np.linalg.norm(side)
        envelope = np.sin(np.pi * s)
        pts = (1 - s)[:, None] * start + s[:, None] * end
        pts += amplitude * envelope[:, None] * (
            np.cos(phase + np.pi * s)[:, None] * bend
            + 0.6 * np.sin(2 * np.pi * s + phase)[:, None] * side
        )
        pts[0] = start
        pts[-1] = end
        graph.add_edge(u, v, key=key, pts=pts)

    graph.graph.update(
        graph_id="getting_started_trivalent_k4",
        input_kind="synthetic_surface_spine",
        is_closed=True,
    )
    return graph


def tube_patch(points, radius=0.11, sides=28):
    tangents = np.gradient(points, axis=0)
    tangents = tangents / np.linalg.norm(tangents, axis=1, keepdims=True)
    reference = np.tile(np.array([0.0, 0.0, 1.0]), (len(points), 1))
    nearly_parallel = np.abs(np.sum(tangents * reference, axis=1)) > 0.92
    reference[nearly_parallel] = np.array([0.0, 1.0, 0.0])
    normals = np.cross(tangents, reference)
    normals = normals / np.linalg.norm(normals, axis=1, keepdims=True)
    binormals = np.cross(tangents, normals)

    theta = np.linspace(0.0, 2 * np.pi, sides, endpoint=True)
    circle = (
        np.cos(theta)[None, :, None] * normals[:, None, :]
        + np.sin(theta)[None, :, None] * binormals[:, None, :]
    )
    tube = points[:, None, :] + radius * circle
    return {"x": tube[:, :, 0], "y": tube[:, :, 1], "z": tube[:, :, 2]}


def sphere_patch(center, radius=0.18, samples=28):
    phi = np.linspace(0.0, np.pi, samples)
    theta = np.linspace(0.0, 2 * np.pi, samples)
    phi, theta = np.meshgrid(phi, theta, indexing="ij")
    return {
        "x": center[0] + radius * np.sin(phi) * np.cos(theta),
        "y": center[1] + radius * np.sin(phi) * np.sin(theta),
        "z": center[2] + radius * np.cos(phi),
    }


def trivalent_k4_surface_graph(tube_radius=0.12):
    graph = trivalent_k4_spine()
    tube_surfaces = [
        tube_patch(data["pts"], radius=tube_radius)
        for _, _, data in graph.edges(data=True)
    ]
    vertex_surfaces = [
        sphere_patch(data["pos"], radius=1.45 * tube_radius)
        for _, data in graph.nodes(data=True)
    ]
    return tube_surfaces, vertex_surfaces, graph


tube_surfaces, vertex_surfaces, graph = trivalent_k4_surface_graph()
print(f"surface tube patches = {len(tube_surfaces)}")
print(f"surface vertex patches = {len(vertex_surfaces)}")
print(f"nodes_edges = {(graph.number_of_nodes(), graph.number_of_edges())}")
print(f"degrees = {dict(graph.degree())}")


### View the finite-thickness geometry

The blue tubes show the edge neighborhoods and the red regions show the branch
neighborhoods.

Use this view to check that the geometry has the connectivity you expect before
reducing it to a graph. The topology calculation below uses the center-line
spatial graph, not the rendered tube thickness itself.


In [ ]:
fig = go.Figure()
for patch in tube_surfaces:
    fig.add_trace(
        go.Surface(
            x=patch["x"],
            y=patch["y"],
            z=patch["z"],
            surfacecolor=np.zeros_like(patch["x"]),
            colorscale=[[0, BLUE], [1, BLUE]],
            showscale=False,
            opacity=0.42,
        )
    )
for patch in vertex_surfaces:
    fig.add_trace(
        go.Surface(
            x=patch["x"],
            y=patch["y"],
            z=patch["z"],
            surfacecolor=np.zeros_like(patch["x"]),
            colorscale=[[0, RED], [1, RED]],
            showscale=False,
            opacity=0.92,
        )
    )
apply_kg_layout(fig).show()


## 1.4 Inspect the spatial graph

Before projecting a graph, check the object you are actually about to analyze.

In the representation used here:

- each node has a three-dimensional `pos`;
- each edge has a sampled three-dimensional polyline `pts`;
- the graph connectivity is stored by NetworkX.

The next cells print the basic graph data and show the embedding. For your own
data, this is where you should catch missing branches, accidental extra edges, or
incorrect vertex positions.


In [ ]:
print(f"nodes_edges = {(graph.number_of_nodes(), graph.number_of_edges())}")
print(f"closed = {graph.graph['is_closed']}")
print(f"degrees = {dict(graph.degree())}")
print("vertices:")
for node, data in graph.nodes(data=True):
    coords = tuple(round(c, 3) for c in data["pos"])
    print(f"  {node}: {coords}")

print("edges:")
for source, target, key, data in graph.edges(keys=True, data=True):
    print(f"  {key}: {source} -> {target}, points={data['pts'].shape}")


In [ ]:
fig = plot_graph_kg(graph)
fig.show()


## 1.5 Optional: apply a repulsive-curve layout

Repulsive layout is an **optional** preprocessing step for improving an
embedding before projection. It requires the separate Repulsor native solver
and is not needed for the rest of this notebook. After installing that
workflow, inspect its command-line options with:

```bash
uv run kg-repulsive-layout --help
```

The dedicated Repulsive Layout guide explains the pinned native dependency,
input/output files, and topology checks. The cells below deliberately continue
with the original `graph`, so the basic projection workflow remains runnable
without the native solver.

This run continues with the original `graph`. If you later produce a relaxed
embedding, validate it and substitute it for `graph` in the same projection and
invariant cells below. Keep both the input and relaxed graphs so the geometric
provenance of the projection remains auditable.

In [ ]:
fig = plot_graph_kg(graph)
fig.show()


## 1.6 Choose a planar projection and inspect the PD code

The Yamada calculation uses a planar diagram of the spatial graph. You therefore
need a valid projection before computing the invariant.

For this introductory example, use a fixed viewing direction so that the output
is reproducible. The returned `projection` object gives you the information you
should inspect:

- `projection.rotation_angles` — the chosen view;
- `projection.num_crossings` — the number of projected crossings;
- `projection.vertices` — projected rigid vertices;
- `projection.crossings` — over/under crossing data;
- `projection.arcs` — planar arc segments;
- `projection.pd_code` — the diagram encoding used by the invariant calculation.

Run the next two cells to print the diagram data and plot the exact projection
that will be used downstream.

All projection figures in this guide use the same visual convention:
**blue edges and red graph vertices**.  At a crossing, the under-strand has a small
visible gap while the over-strand remains continuous, so the figure displays the
over/under information used by the PD code rather than replacing it with a black
crossing marker.


In [ ]:
projection = select_projection(graph, rotation_angles=(0.0, 0.0, 0.0))
print(f"rotation_angles = {tuple(round(a, 2) for a in projection.rotation_angles)}")
print(f"crossings = {projection.num_crossings}")
print(f"pd_code = {projection.pd_code}")


In [ ]:
plot_projection_diagram(
    projection,
    title="Selected planar projection",
)

## 1.7 Compute the Yamada polynomial

Now compute the invariant using the **same graph and the same projection angles**
you just inspected. Setting `return_result=True` returns the polynomial together with the selected projection
metadata, which is useful when you want to keep a reproducible record of the
calculation.


In [ ]:
Y = sp.Symbol("Y")
result = compute_yamada_polynomial(
    graph,
    Y,
    rotation_angles=projection.rotation_angles,
    return_result=True,
    n_jobs=1,
)
print_upsilon("G", result.polynomial)
print(f"selected crossings = {result.projection.num_crossings}")


## 1.8 Continue with the workflow you need

You have now seen the minimum topology pipeline. Continue according to your task:

- **[2. Core Workflows](02_core_workflows.ipynb)** —
  inspect extraction, simplification, minimum-edge contraction, smoothing, projection
  selection, and PD-code construction in detail.
- **[Physics Applications](applications/01_physics_applications.ipynb)** — start from
  Hamiltonians, exceptional/nodal structures, physical fields, parameter sweeps, or materials.
- **[Mathematics Applications](applications/02_mathematics_applications.ipynb)** — work
  directly with graph families, exact invariants, and structured Yamada datasets.
- **[Analytic Knot Fields](applications/04_analytic_knot_fields.ipynb)** — start from
  a named knot/link, braid word, analytic complex field, or a knot-field deformation.
- **Protein inputs** — the public PDB/mmCIF backbone adapters load ordered
  backbone geometry; domain-specific protein-to-spatial-graph modeling remains
  a separate application workflow.
- **[3. Advanced & Reproduction](03_advanced_and_reproduction.ipynb)** — test
  sensitivity, projection robustness, state expansions, and larger examples.
